# Plan D: AlphaZero 風 value NN 学習 (Colab T4 GPU)

MCTS rollout で計算した P(win|state) を target に value NN を学習。
v1-v5 の「最終勝者 ±1」 失敗を踏まえ、 critical state でも正確な評価関数を NN 化。

**事前準備**:
1. ローカルで `scripts/collect_mcts_rollout_snapshots.py` 実行 → snapshot.jsonl 生成
2. snapshot を Drive `MyDrive/onepiece_nn/mcts_rollout_snapshots.jsonl` にアップロード
3. このノートブックを開く、 ランタイム > T4 GPU、 全セル実行
4. `MyDrive/onepiece_nn/value_nn_alphazero.pt` を ローカル `db/value_nn_alphazero.pt` に download

In [ ]:
import torch
print('torch:', torch.__version__)
print('CUDA:', torch.cuda.is_available())
assert torch.cuda.is_available()

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os
WORK_DIR = '/content/drive/MyDrive/onepiece_nn'
SNAP_PATH = os.path.join(WORK_DIR, 'mcts_rollout_snapshots.jsonl')
OUT_PT = os.path.join(WORK_DIR, 'value_nn_alphazero.pt')
assert os.path.exists(SNAP_PATH)
print('snapshot size:', os.path.getsize(SNAP_PATH) // (1024*1024), 'MB')

In [ ]:
import json, time, numpy as np
STATE_DIM = 172
snapshots = []
with open(SNAP_PATH) as f:
    for line in f:
        try:
            s = json.loads(line)
        except: continue
        if 'state_encoded' in s and 'p_win' in s:
            snapshots.append(s)
print(f'loaded {len(snapshots)} snapshots')
p_vals = [s['p_win'] for s in snapshots]
print(f'p_win: mean={np.mean(p_vals):.3f}, std={np.std(p_vals):.3f}')

X = np.zeros((len(snapshots), STATE_DIM), dtype=np.float32)
Y = np.zeros((len(snapshots),), dtype=np.float32)
for i, s in enumerate(snapshots):
    se = s['state_encoded']
    X[i, :min(STATE_DIM, len(se))] = se[:STATE_DIM]
    Y[i] = s['p_win']
device = torch.device('cuda')
X_t = torch.from_numpy(X).to(device)
Y_t = torch.from_numpy(Y).to(device)
n = len(X_t); split = int(n * 0.85)
X_train, X_val = X_t[:split], X_t[split:]
Y_train, Y_val = Y_t[:split], Y_t[split:]
print(f'train: {len(X_train)}, val: {len(X_val)}')

In [ ]:
import torch.nn as nn

class ValueNN(nn.Module):
    def __init__(self, input_dim=172, hidden=256, dropout=0.15):
        super().__init__()
        self.body = nn.Sequential(
            nn.Linear(input_dim, hidden), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(hidden, hidden), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(hidden, hidden // 2), nn.ReLU(), nn.Dropout(dropout),
        )
        self.head = nn.Linear(hidden // 2, 1)
    def forward(self, x):
        return torch.sigmoid(self.head(self.body(x))).squeeze(-1)

model = ValueNN().to(device)
print(f'params: {sum(p.numel() for p in model.parameters())}')

In [ ]:
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader
EPOCHS = 50
BATCH = 1024
LR = 1e-3
optimizer = optim.Adam(model.parameters(), lr=LR)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=1e-5)
train_ds = TensorDataset(X_train, Y_train)
train_loader = DataLoader(train_ds, batch_size=BATCH, shuffle=True)

best_val = float('inf')
for epoch in range(EPOCHS):
    model.train()
    tloss = 0.0; nb = 0
    for xb, yb in train_loader:
        optimizer.zero_grad()
        pred = model(xb)
        loss = ((pred - yb) ** 2).mean()  # MSE on probability
        loss.backward()
        optimizer.step()
        tloss += loss.item(); nb += 1
    scheduler.step()
    model.eval()
    with torch.no_grad():
        vpred = model(X_val)
        vloss = ((vpred - Y_val) ** 2).mean().item()
        # sign accuracy: P(win) > 0.5 が真の勝者と一致する率
        vsign = ((vpred > 0.5) == (Y_val > 0.5)).float().mean().item()
    print(f'  epoch {epoch+1}/{EPOCHS}: train_loss={tloss/nb:.4f}, val_loss={vloss:.4f}, val_sign_acc={vsign:.3f}')
    if vloss < best_val:
        best_val = vloss
        torch.save(model.state_dict(), OUT_PT)

print(f'\nbest val_loss: {best_val:.4f}, saved to {OUT_PT}')